<a href="https://colab.research.google.com/github/varun0852/for-study/blob/master/Spam_%26_Non_Spam_with_Passing_Data_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import string
import pickle
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import pandas as pd

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\razee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
def preprocess_text(text):
    no_punctuation = [char for char in text if char not in string.punctuation]
    no_punctuation = ''.join(no_punctuation)
    words = no_punctuation.lower().split()

    stop_words = set(stopwords.words('english'))
    ps = PorterStemmer()

    processed_words = [ps.stem(word) for word in words if word not in stop_words]
    return ' '.join(processed_words)

In [ ]:
data = pd.read_csv('D:/Course Files/Datasets/NB.csv', encoding='ISO-8859-1')
data.columns = ['label', 'text']

In [ ]:
data

,label,text
0,ham,Hope you are having a good week. Just checking in
1,ham,K..give back my thanks.
2,ham,Am also doing in cbe only. But have to pay.
3,spam,"complimentary 4 STAR Ibiza Holiday or å£10,000..."
4,spam,okmail: Dear Dave this is your final notice to...
...,...,...
5554,ham,You are a great role model. You are giving so ...
5555,ham,"Awesome, I remember the last time we got someb..."
5556,spam,"If you don't, your prize will go to another cu..."
5557,spam,"SMS. ac JSco: Energy is high, but u may not kn..."


In [ ]:
data['text'] = data['text'].apply(preprocess_text)

In [ ]:
data['text']

0                                    hope good week check
1                                        kgive back thank
2                                            also cbe pay
3       complimentari 4 star ibiza holiday å£10000 cas...
4       okmail dear dave final notic collect 4 tenerif...
                              ...                        
5554    great role model give much realli wish day mir...
5555    awesom rememb last time got somebodi high firs...
5556    dont prize go anoth custom tc wwwtcbiz 18 150p...
5557    sm ac jsco energi high u may know 2channel 2da...
5558                                 shall call dear food
Name: text, Length: 5559, dtype: object

In [ ]:
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [ ]:
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       963
           1       0.99      0.74      0.85       149

    accuracy                           0.96      1112
   macro avg       0.98      0.87      0.91      1112
weighted avg       0.97      0.96      0.96      1112



In [ ]:
with open('D:/Course Files/Script Files/Virtusa/spam_classifier.pkl', 'wb') as file:
    pickle.dump(model, file)

with open('D:/Course Files/Script Files/Virtusa/vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)


In [ ]:
def preprocess_text(text):
    no_punctuation = [char for char in text if char not in string.punctuation]
    no_punctuation = ''.join(no_punctuation)
    words = no_punctuation.lower().split()

    stop_words = set(stopwords.words('english'))
    ps = PorterStemmer()

    processed_words = [ps.stem(word) for word in words if word not in stop_words]
    return ' '.join(processed_words)


with open('D:/Course Files/Script Files/Virtusa/vectorizer.pkl', 'rb') as file:
    vectorizer = pickle.load(file)

with open('D:/Course Files/Script Files/Virtusa/spam_classifier.pkl', 'rb') as file:
    model = pickle.load(file)


def classify_message(message):
    preprocessed_message = preprocess_text(message)
    message_vector = vectorizer.transform([preprocessed_message])
    prediction = model.predict(message_vector)
    return 'Spam' if prediction[0] == 1 else 'Not Spam'


message = input("Enter a message to classify: ")
result = classify_message(message)
print(f'The message is: {result}')

Enter a message to classify: I got the prize money of 5000 %
The message is: Not Spam


In [ ]:
message = input("Enter a message to classify: ")
result = classify_message(message)
print(f'The message is: {result}')

Enter a message to classify: Claim
The message is: Spam
